# FINS 实验流水线

一条 notebook 串起 4 个模块（原 notebook 已 py 化）：

| 步骤       | 模块 | 作用                                                                                           |
|------------|---|------------------------------------------------------------------------------------------------|
| 1 生成配置 | `_load.py` | 按拓扑 / 时序 / 负载随机生成 pipeline cfg → `pipeline/*.json`                             |
| 2 采集     | `_test.py` | 【fins 测试专用】每份 cfg 起 `bin/client` + `bin/server` 跑 `dur_s` 秒 → trace 复制到 `result/`              |
| 3 标准化   | `std.py` | 【fins 测试专用】 用 JSON 的执行用时在 `execute→complete` 内插 `working` 行 → `result_std/` |
| 4 分析画图 | `plot.py` | 核心甘特、核利用率、生命周期分布                                                               |

> 三个数据目录都在**仓库根**：`pipeline/`（配置）、`result/`（原始 trace）、`result_std/`（标准化 trace）。
> 下面所有相对路径都按仓库根解析，不依赖 notebook 的 cwd。

In [3]:
import importlib

# 导入你的基础模块与工具库
import _load
import _test
import _lttng2timeline
import _lttng2preempt

# 导入绘图模块
import _csv2overhead
import _csv2timeline
from _csv2overhead import analyze_cpu_utilization
from _csv2timeline import generate_execution_gantt

# 将所有需要支持热重载的本地模块统一放到元组中刷新
for m in (_load, _test, _csv2timeline, _csv2overhead):
  importlib.reload(m)

## 1. 生成配置（`_load.py`）

5 种拓扑（multihop / fork / join / feedback / mixed，均可混入 acc 的 hist 窗口读）
+ 时序（`ptimed`、`period_divisors`）+ 负载（`u`、`H_ms` 等）随机生成，
文件名 `<kind>_u<u>_m<m>_ms<桶>_s<seed>.json`。

In [ ]:
my_cfg_dir = "tool/pipeline"
CONFIG = {

    "topology": {

        "multihop": {
            "paths": (1, 4),
            "depth": (2, 8),
        },

        "fork": {
            "fan": (2, 8),
            "bdepth": (1, 3),
        },

        "join": {
            "fan": (2, 8),
            "bdepth": (1, 3),
            "tail": (1, 3),
        },

        "feedback": {
            "depth": (3, 8),
            "histN": (3, 8),
        },

        "mixed": {
            "nseg": (3, 8),

            "chain_prob": 0.45,
            "fork_join_prob": 0.30,
            "feedback_prob": 0.25,
        },
    },

    "temporal": {

        # Probability that a non-source node is timed.
        "ptimed": 0.35,

        # T values are H / divisor.
        #
        # Example:
        #
        # H=100
        #
        # divisor 1 -> 100 ms
        # divisor 2 -> 50 ms
        # divisor 4 -> 25 ms
        #
        "period_divisors": [1, 2, 4, 5, 10, 20],
    },

    "workload": {

        "u": [
            0.1,
            0.3,
            0.7,
            0.9,
        ],

        "workers": [
            1,
            2,
            3,
        ],

        # Hyperperiod.
        "H_ms": 100,

        # ----------------------------------------------------
        # Makespan classification.
        #
        # Example H=100, width=5:
        #
        # bin 00: [0,5)
        # bin 01: [5,10)
        # ...
        # bin 18: [90,95)
        # bin 19: [95,100]
        #
        # The last bucket includes H.
        # ----------------------------------------------------

        "makespan_bin_width_ms": 10.0,

        # Anything above this goes into overflow.
        #
        # None:
        #     use H_ms.
        #
        "makespan_max_ms": 80,

        # Number of final samples per bucket.
        "n_per": 5,

        # ----------------------------------------------------
        # Generation policy.
        # ----------------------------------------------------

        # Initial candidate generation.
        "initial_attempts": 500,

        # Extra attempts for incomplete buckets.
        "refill_attempts": 500,

        # Maximum number of refill rounds.
        "max_refill_rounds": 1,

        # Keep at most:
        #
        #     n_per * candidate_factor
        #
        # candidates per bucket.
        #
        "candidate_factor": 1,

        # Random seed.
        "seed_base": 20260910,

    },

    "solver": {

        # Utilization numerical tolerance.
        "u_tolerance": 1e-8,

        # Minimum WCET.
        #
        # Plugin cfg is in microseconds.
        #
        "min_wcet_us": 100,

        # C_i <= T_i * max_c_ratio
        "max_c_ratio": 1.0,

        # Number of attempts used to find a C allocation.
        "c_attempts": 300,
    },
}

_load.generate_all(
    out_dir=my_cfg_dir,
    config=CONFIG
)

## 2. 采集（`_test.py`）

对每份 cfg：起 `client` + 发配置 + 跑 `dur_s` 秒 → 终止 → 把 `tool/temp/tracing.csv`
复制成 `result/<cfg名>.csv`（原件保留，只搬原始 trace，不算指标）。

正式实验要走独占核：`cores="1-6"` 会改用 `sudo tool/client.sh <cores> <workers>`（需要 root）。

In [23]:
# 指定你的目录
my_cfg_dir = "tool/pipeline"
my_result_dir = "tool/result_100ms"

# 一键运行（cpu_offset=1 代表核心从 1 开始排，如果 m=3 就会自动分配核 "1-3"）
_test.run_all(
    target_cfg_dir=my_cfg_dir,
    target_result_dir=my_result_dir,
    run_time=2.0,     # 运行时间
    cpu_offset=1      # 起始核号（避开核心 0 给系统）
)

🔒 该评测脚本需要 root 权限来配置 Cgroup 与绑定核心
🚀 开始智能批量评测（自动从文件名匹配 m）
   📂 输入配置目录: tool/pipeline
   📁 结果输出目录: tool/result_100ms
   📊 发现测试用例: 5 个

进度 [1/5]: feedback_u70_m3_ms05_s20632672.json

>>> [开始测试] feedback_u70_m3_ms05_s20632672 (自动解析: m=3 -> workers=3, cores=1-3)
  [1/5] 创建并启动 LTTng 追踪会话 (fins_eval_feedback_u70_m3_ms05_s20632672_000343)...
  [2/5] 启动 Client 进程 (独占核 1-3, workers=3)...
  [3/5] 等待客户端预热 2.0s 并推送配置...
  [Server] 配置灌入成功，持续运行 2.0s...
  [4/5] 停止并销毁 LTTng 追踪会话...
  [5/5] 关闭清理 Client 进程 (pgid=22259)...
  [成功] 轨迹数据已归档至: /home/jenny/Documents/GitHub/fins/tool/result_100ms/feedback_u70_m3_ms05_s20632672_000343/trace
  [验证] ✅ 成功捕捉到 1690 条目标跟踪事件！

进度 [2/5]: fork_u70_m3_ms05_s20984935.json

>>> [开始测试] fork_u70_m3_ms05_s20984935 (自动解析: m=3 -> workers=3, cores=1-3)
  [1/5] 创建并启动 LTTng 追踪会话 (fins_eval_fork_u70_m3_ms05_s20984935_000351)...
  [2/5] 启动 Client 进程 (独占核 1-3, workers=3)...
  [3/5] 等待客户端预热 2.0s 并推送配置...
  [Server] 配置灌入成功，持续运行 2.0s...
  [4/5] 停止并销毁 LTTng 追踪会话...
  [5/5] 关闭清理 Client 进程 

# 3. 转义（tran）
1. lttng 2 timeline
2. lttng 2 overhead

In [24]:
_lttng2timeline.run_export(
    results_dir="./result_100ms",
    outdir="./result_100ms",
    after_us=2000 * 1000 # 过滤前 5 秒预热
)

# 2. 批量导出抢占与切换数据（仅保留 worker 线程）
_lttng2preempt.run_export(
    results_dir="./result_100ms",
    outdir="./result_100ms",
    after_us=2000 * 1000,
    filter_worker_only=True
)

开始批量导出 timeline: 5 个实验 -> ./result_100ms/
  [成功] feedback_u70_m3_ms05_s20632672_000343 -> feedback_u70_m3_ms05_s20632672_000343_timeline.csv (907 行)
  [成功] fork_u70_m3_ms05_s20984935_000351 -> fork_u70_m3_ms05_s20984935_000351_timeline.csv (750 行)
  [成功] join_u70_m3_ms07_s20628568_000359 -> join_u70_m3_ms07_s20628568_000359_timeline.csv (2197 行)
  [成功] mixed_u70_m3_ms05_s30924190_000407 -> mixed_u70_m3_ms05_s30924190_000407_timeline.csv (1325 行)
  [成功] multihop_u70_m3_ms05_s20633261_000415 -> multihop_u70_m3_ms05_s20633261_000415_timeline.csv (904 行)
完成。成功 5 / 5
开始处理抢占数据: 5 个实验 -> ./result_100ms/
  [成功] feedback_u70_m3_ms05_s20632672_000343 -> feedback_u70_m3_ms05_s20632672_000343_preempt.csv (1209 rows)
  [成功] fork_u70_m3_ms05_s20984935_000351 -> fork_u70_m3_ms05_s20984935_000351_preempt.csv (1483 rows)
  [成功] join_u70_m3_ms07_s20628568_000359 -> join_u70_m3_ms07_s20628568_000359_preempt.csv (1125 rows)
  [成功] mixed_u70_m3_ms05_s30924190_000407 -> mixed_u70_m3_ms05_s30924190_000407_pr

## 4. 分析与画图（`plot.py`）



In [ ]:
# typical/CIE_FIFO_IPC/
# feedback_u70_m3_ms05_s20632672_135004_preempt.csv
# feedback_u70_m3_ms05_s20632672_135004_timeline.csv
# fork_u70_m3_ms05_s20984935_135017_preempt.csv
# fork_u70_m3_ms05_s20984935_135017_timeline.csv
# join_u70_m3_ms07_s20628568_135030_preempt.csv
# join_u70_m3_ms07_s20628568_135030_timeline.csv
# mixed_u70_m3_ms05_s30924190_135042_preempt.csv
# mixed_u70_m3_ms05_s30924190_135042_timeline.csv
# multihop_u70_m3_ms05_s20633261_135055_preempt.csv
# multihop_u70_m3_ms05_s20633261_135055_timeline.csv

In [ ]:
# typical/CIE_FIFO_!IPC/
# feedback_u70_m3_ms05_s20632672_133705_preempt.csv
# feedback_u70_m3_ms05_s20632672_133705_timeline.csv
# fork_u70_m3_ms05_s20984935_133719_preempt.csv
# fork_u70_m3_ms05_s20984935_133719_timeline.csv
# join_u70_m3_ms07_s20628568_133732_preempt.csv
# join_u70_m3_ms07_s20628568_133732_timeline.csv
# mixed_u70_m3_ms05_s30924190_133745_preempt.csv
# mixed_u70_m3_ms05_s30924190_133745_timeline.csv
# multihop_u70_m3_ms05_s20633261_133758_preempt.csv
# multihop_u70_m3_ms05_s20633261_133758_timeline.csv

In [ ]:
# typical/CIE_RR_!IPC/
# feedback_u70_m3_ms05_s20632672_135045_preempt.csv
# feedback_u70_m3_ms05_s20632672_135045_timeline.csv
# fork_u70_m3_ms05_s20984935_135058_preempt.csv
# fork_u70_m3_ms05_s20984935_135058_timeline.csv
# join_u70_m3_ms07_s20628568_135111_preempt.csv
# join_u70_m3_ms07_s20628568_135111_timeline.csv
# mixed_u70_m3_ms05_s30924190_135124_preempt.csv
# mixed_u70_m3_ms05_s30924190_135124_timeline.csv
# multihop_u70_m3_ms05_s20633261_135136_preempt.csv
# multihop_u70_m3_ms05_s20633261_135136_timeline.csv

In [ ]:
# typical/CIE_RR_IPC/
# feedback_u70_m3_ms05_s20632672_135906_preempt.csv
# feedback_u70_m3_ms05_s20632672_135906_timeline.csv
# fork_u70_m3_ms05_s20984935_135919_preempt.csv
# fork_u70_m3_ms05_s20984935_135919_timeline.csv
# join_u70_m3_ms07_s20628568_135932_preempt.csv
# join_u70_m3_ms07_s20628568_135932_timeline.csv
# mixed_u70_m3_ms05_s30924190_135945_preempt.csv
# mixed_u70_m3_ms05_s30924190_135945_timeline.csv
# multihop_u70_m3_ms05_s20633261_135957_preempt.csv
# multihop_u70_m3_ms05_s20633261_135957_timeline.csv

In [ ]:
# typical/CIE_OTHER_IPC/
# feedback_u70_m3_ms05_s20632672_140406_preempt.csv
# feedback_u70_m3_ms05_s20632672_140406_timeline.csv
# fork_u70_m3_ms05_s20984935_140419_preempt.csv
# fork_u70_m3_ms05_s20984935_140419_timeline.csv
# join_u70_m3_ms07_s20628568_140431_preempt.csv
# join_u70_m3_ms07_s20628568_140431_timeline.csv
# mixed_u70_m3_ms05_s30924190_140444_preempt.csv
# mixed_u70_m3_ms05_s30924190_140444_timeline.csv
# multihop_u70_m3_ms05_s20633261_140457_preempt.csv
# multihop_u70_m3_ms05_s20633261_140457_timeline.csv

In [ ]:
# typical/CIE_OTHER_!IPC/
# feedback_u70_m3_ms05_s20632672_141144_preempt.csv
# feedback_u70_m3_ms05_s20632672_141144_timeline.csv
# fork_u70_m3_ms05_s20984935_141157_preempt.csv
# fork_u70_m3_ms05_s20984935_141157_timeline.csv
# join_u70_m3_ms07_s20628568_141209_preempt.csv
# join_u70_m3_ms07_s20628568_141209_timeline.csv
# mixed_u70_m3_ms05_s30924190_141222_preempt.csv
# mixed_u70_m3_ms05_s30924190_141222_timeline.csv
# multihop_u70_m3_ms05_s20633261_141235_preempt.csv
# multihop_u70_m3_ms05_s20633261_141235_timeline.csv

In [ ]:
# typical/MTE_IPC
# feedback_u70_m3_ms05_s20632672_141727_preempt.csv
# feedback_u70_m3_ms05_s20632672_141727_timeline.csv
# fork_u70_m3_ms05_s20984935_141740_preempt.csv
# fork_u70_m3_ms05_s20984935_141740_timeline.csv
# join_u70_m3_ms07_s20628568_141753_preempt.csv
# join_u70_m3_ms07_s20628568_141753_timeline.csv
# mixed_u70_m3_ms05_s30924190_141806_preempt.csv
# mixed_u70_m3_ms05_s30924190_141806_timeline.csv
# multihop_u70_m3_ms05_s20633261_141819_preempt.csv
# multihop_u70_m3_ms05_s20633261_141819_timeline.csv

In [ ]:
# typical/MTE_!IPC
# feedback_u70_m3_ms05_s20632672_142400_preempt.csv
# feedback_u70_m3_ms05_s20632672_142400_timeline.csv
# fork_u70_m3_ms05_s20984935_142412_preempt.csv
# fork_u70_m3_ms05_s20984935_142412_timeline.csv
# join_u70_m3_ms07_s20628568_142425_preempt.csv
# join_u70_m3_ms07_s20628568_142425_timeline.csv
# mixed_u70_m3_ms05_s30924190_142438_preempt.csv
# mixed_u70_m3_ms05_s30924190_142438_timeline.csv
# multihop_u70_m3_ms05_s20633261_142451_preempt.csv
# multihop_u70_m3_ms05_s20633261_142451_timeline.csv

In [ ]:
# typical/FINS
# feedback_u70_m3_ms05_s20632672_142727_preempt.csv
# feedback_u70_m3_ms05_s20632672_142727_timeline.csv
# fork_u70_m3_ms05_s20984935_142737_preempt.csv
# fork_u70_m3_ms05_s20984935_142737_timeline.csv
# join_u70_m3_ms07_s20628568_142746_preempt.csv
# join_u70_m3_ms07_s20628568_142746_timeline.csv
# mixed_u70_m3_ms05_s30924190_142754_preempt.csv
# mixed_u70_m3_ms05_s30924190_142754_timeline.csv
# multihop_u70_m3_ms05_s20633261_142803_preempt.csv
# multihop_u70_m3_ms05_s20633261_142803_timeline.csv

In [25]:
# 1. 绘制甘特图
fig_gantt_CIE_FIFO_IPC = generate_execution_gantt(
    preempt_csv = "typical/CIE_FIFO_IPC/feedback_u70_m3_ms05_s20632672_135004_preempt.csv",
    timeline_csv = "typical/CIE_FIFO_IPC/feedback_u70_m3_ms05_s20632672_135004_timeline.csv",
    zoom_window_ms=[0, 1000])

fig_gantt_CIE_FIFO_noIPC = generate_execution_gantt(
    preempt_csv = "typical/CIE_FIFO_!IPC/feedback_u70_m3_ms05_s20632672_133705_preempt.csv",
    timeline_csv = "typical/CIE_FIFO_!IPC/feedback_u70_m3_ms05_s20632672_133705_timeline.csv",
    zoom_window_ms=[0, 1000])

fig_gantt_CIE_RR_IPC = generate_execution_gantt(
    preempt_csv = "typical/CIE_RR_IPC/feedback_u70_m3_ms05_s20632672_135906_preempt.csv",
    timeline_csv = "typical/CIE_RR_IPC/feedback_u70_m3_ms05_s20632672_135906_timeline.csv",
    zoom_window_ms=[0, 1000])

fig_gantt_CIE_RR_noIPC = generate_execution_gantt(
    preempt_csv = "typical/CIE_RR_!IPC/feedback_u70_m3_ms05_s20632672_135045_preempt.csv",
    timeline_csv = "typical/CIE_RR_!IPC/feedback_u70_m3_ms05_s20632672_135045_timeline.csv",
    zoom_window_ms=[0, 1000])

fig_gantt_CIE_OTHER_IPC = generate_execution_gantt(
    preempt_csv = "typical/CIE_OTHER_IPC/feedback_u70_m3_ms05_s20632672_140406_preempt.csv",
    timeline_csv = "typical/CIE_OTHER_IPC/feedback_u70_m3_ms05_s20632672_140406_timeline.csv",
    zoom_window_ms=[0, 1000])

fig_gantt_CIE_OTHER_noIPC = generate_execution_gantt(
    preempt_csv = "typical/CIE_OTHER_!IPC/feedback_u70_m3_ms05_s20632672_141144_preempt.csv",
    timeline_csv = "typical/CIE_OTHER_!IPC/feedback_u70_m3_ms05_s20632672_141144_timeline.csv",
    zoom_window_ms=[0, 1000])

fig_gantt_MTE_IPC = generate_execution_gantt(
    preempt_csv = "typical/MTE_IPC/feedback_u70_m3_ms05_s20632672_141727_preempt.csv",
    timeline_csv = "typical/MTE_IPC/feedback_u70_m3_ms05_s20632672_141727_timeline.csv",
    zoom_window_ms=[0, 1000])

fig_gantt_MTE_noIPC = generate_execution_gantt(
    preempt_csv = "typical/MTE_!IPC/feedback_u70_m3_ms05_s20632672_142400_preempt.csv",
    timeline_csv = "typical/MTE_!IPC/feedback_u70_m3_ms05_s20632672_142400_timeline.csv",
    zoom_window_ms=[0, 1000])

fig_gantt_FINS_1ms = generate_execution_gantt(
    preempt_csv = "typical/result_1ms/feedback_u70_m3_ms05_s20632672_142727_preempt.csv",
    timeline_csv = "typical/result_1ms/feedback_u70_m3_ms05_s20632672_142727_timeline.csv",
    zoom_window_ms=[0, 1000])

# 100ms
# feedback_u70_m3_ms05_s20632672_000343_preempt.csv
# feedback_u70_m3_ms05_s20632672_000343_timeline.csv
fig_gantt_FINS_100ms = generate_execution_gantt(
    preempt_csv = "typical/result_100ms/feedback_u70_m3_ms05_s20632672_000343_preempt.csv",
    timeline_csv = "typical/result_100ms/feedback_u70_m3_ms05_s20632672_000343_timeline.csv",
    zoom_window_ms=[0, 1000])

fig_gantt_FINS_1000ms = generate_execution_gantt(
    preempt_csv = "typical/result_1000ms/feedback_u70_m3_ms05_s20632672_215849_preempt.csv",
    timeline_csv = "typical/result_1000ms/feedback_u70_m3_ms05_s20632672_215849_timeline.csv",
    zoom_window_ms=[0, 1000])

fig_gantt_CIE_FIFO_IPC.show()
fig_gantt_CIE_FIFO_noIPC.show()
fig_gantt_CIE_RR_IPC.show()
fig_gantt_CIE_RR_noIPC.show()
fig_gantt_CIE_OTHER_IPC.show()
fig_gantt_CIE_OTHER_noIPC.show()
fig_gantt_MTE_IPC.show()
fig_gantt_MTE_noIPC.show()
fig_gantt_FINS_1ms.show()
fig_gantt_FINS_100ms.show()
fig_gantt_FINS_1000ms.show()



正在读取数据 (Gantt)...
甘特图已保存至: cpu_activate_timeline.html
正在读取数据 (Gantt)...
甘特图已保存至: cpu_activate_timeline.html
正在读取数据 (Gantt)...
甘特图已保存至: cpu_activate_timeline.html
正在读取数据 (Gantt)...
甘特图已保存至: cpu_activate_timeline.html
正在读取数据 (Gantt)...
甘特图已保存至: cpu_activate_timeline.html
正在读取数据 (Gantt)...
甘特图已保存至: cpu_activate_timeline.html
正在读取数据 (Gantt)...
甘特图已保存至: cpu_activate_timeline.html
正在读取数据 (Gantt)...
甘特图已保存至: cpu_activate_timeline.html
正在读取数据 (Gantt)...
甘特图已保存至: cpu_activate_timeline.html
正在读取数据 (Gantt)...
甘特图已保存至: cpu_activate_timeline.html
正在读取数据 (Gantt)...
甘特图已保存至: cpu_activate_timeline.html


In [8]:
# 1. 绘制甘特图
fig_util_CIE_FIFO_IPC = analyze_cpu_utilization(
    preempt_csv = "typical/CIE_FIFO_IPC/feedback_u70_m3_ms05_s20632672_135004_preempt.csv",
    timeline_csv = "typical/CIE_FIFO_IPC/feedback_u70_m3_ms05_s20632672_135004_timeline.csv",
    analysis_window_ms=[50, 900])

fig_util_CIE_FIFO_noIPC = analyze_cpu_utilization(
    preempt_csv = "typical/CIE_FIFO_!IPC/feedback_u70_m3_ms05_s20632672_133705_preempt.csv",
    timeline_csv = "typical/CIE_FIFO_!IPC/feedback_u70_m3_ms05_s20632672_133705_timeline.csv",
    analysis_window_ms=[50, 900])

fig_util_CIE_RR_IPC = analyze_cpu_utilization(
    preempt_csv = "typical/CIE_RR_IPC/feedback_u70_m3_ms05_s20632672_135906_preempt.csv",
    timeline_csv = "typical/CIE_RR_IPC/feedback_u70_m3_ms05_s20632672_135906_timeline.csv",
    analysis_window_ms=[50, 900])

fig_util_CIE_RR_noIPC = analyze_cpu_utilization(
    preempt_csv = "typical/CIE_RR_!IPC/feedback_u70_m3_ms05_s20632672_135045_preempt.csv",
    timeline_csv = "typical/CIE_RR_!IPC/feedback_u70_m3_ms05_s20632672_135045_timeline.csv",
    analysis_window_ms=[50, 900])

fig_util_CIE_OTHER_IPC = analyze_cpu_utilization(
    preempt_csv = "typical/CIE_OTHER_IPC/feedback_u70_m3_ms05_s20632672_140406_preempt.csv",
    timeline_csv = "typical/CIE_OTHER_IPC/feedback_u70_m3_ms05_s20632672_140406_timeline.csv",
    analysis_window_ms=[50, 900])

fig_util_CIE_OTHER_noIPC = analyze_cpu_utilization(
    preempt_csv = "typical/CIE_OTHER_!IPC/feedback_u70_m3_ms05_s20632672_141144_preempt.csv",
    timeline_csv = "typical/CIE_OTHER_!IPC/feedback_u70_m3_ms05_s20632672_141144_timeline.csv",
    analysis_window_ms=[50, 900])

fig_util_MTE_IPC = analyze_cpu_utilization(
    preempt_csv = "typical/MTE_IPC/feedback_u70_m3_ms05_s20632672_141727_preempt.csv",
    timeline_csv = "typical/MTE_IPC/feedback_u70_m3_ms05_s20632672_141727_timeline.csv",
    analysis_window_ms=[50, 900])

fig_util_MTE_noIPC = analyze_cpu_utilization(
    preempt_csv = "typical/MTE_!IPC/feedback_u70_m3_ms05_s20632672_142400_preempt.csv",
    timeline_csv = "typical/MTE_!IPC/feedback_u70_m3_ms05_s20632672_142400_timeline.csv",
    analysis_window_ms=[50, 900])

fig_util_CIE_FIFO_IPC.show()
fig_util_CIE_FIFO_noIPC.show()
fig_util_CIE_RR_IPC.show()
fig_util_CIE_RR_noIPC.show()
fig_util_CIE_OTHER_IPC.show()
fig_util_CIE_OTHER_noIPC.show()
fig_util_MTE_IPC.show()
fig_util_MTE_noIPC.show()

正在读取数据并计算利用率占比...
利用率占比图已保存至: cpu_utilization_breakdown.html
正在读取数据并计算利用率占比...
利用率占比图已保存至: cpu_utilization_breakdown.html
正在读取数据并计算利用率占比...
利用率占比图已保存至: cpu_utilization_breakdown.html
正在读取数据并计算利用率占比...
利用率占比图已保存至: cpu_utilization_breakdown.html
正在读取数据并计算利用率占比...
利用率占比图已保存至: cpu_utilization_breakdown.html
正在读取数据并计算利用率占比...
利用率占比图已保存至: cpu_utilization_breakdown.html
正在读取数据并计算利用率占比...
利用率占比图已保存至: cpu_utilization_breakdown.html
正在读取数据并计算利用率占比...
利用率占比图已保存至: cpu_utilization_breakdown.html


In [29]:
# 1ms
# feedback_u70_m3_ms05_s20632672_142727_preempt.csv
# feedback_u70_m3_ms05_s20632672_142727_timeline.csv

fig_util_FINS_1ms = analyze_cpu_utilization(
    preempt_csv = "typical/result_1ms/feedback_u70_m3_ms05_s20632672_142727_preempt.csv",
    timeline_csv = "typical/result_1ms/feedback_u70_m3_ms05_s20632672_142727_timeline.csv",
    analysis_window_ms=[50, 1000])

# 100ms
# feedback_u70_m3_ms05_s20632672_000343_preempt.csv
# feedback_u70_m3_ms05_s20632672_000343_timeline.csv
fig_util_FINS_100ms = analyze_cpu_utilization(
    preempt_csv = "typical/result_100ms/feedback_u70_m3_ms05_s20632672_000343_preempt.csv",
    timeline_csv = "typical/result_100ms/feedback_u70_m3_ms05_s20632672_000343_timeline.csv",
    analysis_window_ms=[50, 1000])

# 1000ms
# feedback_u70_m3_ms05_s20632672_215849_preempt.csv
# feedback_u70_m3_ms05_s20632672_215849_timeline.csv

fig_util_FINS_1000ms = analyze_cpu_utilization(
    preempt_csv = "typical/result_1000ms/feedback_u70_m3_ms05_s20632672_215849_preempt.csv",
    timeline_csv = "typical/result_1000ms/feedback_u70_m3_ms05_s20632672_215849_timeline.csv",
    analysis_window_ms=[50, 3000])

fig_util_FINS_1ms.show()
fig_util_FINS_100ms.show()
fig_util_FINS_1000ms.show()

正在读取数据并计算利用率占比...
利用率占比图已保存至: cpu_utilization_breakdown.html
正在读取数据并计算利用率占比...
利用率占比图已保存至: cpu_utilization_breakdown.html
正在读取数据并计算利用率占比...
利用率占比图已保存至: cpu_utilization_breakdown.html
